# Day 2 Exercise: Webpage Summarizer with Ollama

Upgrade of the day 1 project: summarize a webpage using an open-source model running locally via Ollama instead of OpenAI.

Benefits: no API charges, and data doesn't leave the machine.
Disadvantage: less powerful than a frontier model.

In [ ]:
# imports

import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import Markdown, display

# Check that Ollama is running

print(requests.get("http://localhost:11434").content)


def fetch_website_contents(url, max_chars=2000):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers, timeout=15)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for tag in soup.body(["script", "style", "img", "input"]):
            tag.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:max_chars]

In [ ]:
# Ollama exposes an OpenAI compatible endpoint, so we reuse the OpenAI client

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2"  # use "llama3.2:1b" if your machine is slow

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
system_prompt = """
You are an assistant that analyzes the contents of a website
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://edwarddonner.com")

In [ ]:
display_summary("https://anthropic.com")

In [ ]:
subject_system_prompt = """
You are an assistant that writes email subject lines.
Given the body of an email, suggest one clear, specific subject line.
Keep it under 10 words, avoid clickbait and all caps, and capture the main purpose or action required.
Respond with only the subject line, with no quotes and no extra commentary.
"""

def suggest_subject(email_body):
    messages = [
        {"role": "system", "content": subject_system_prompt},
        {"role": "user", "content": "Here is the email body:\n\n" + email_body}
    ]
    response = ollama.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content.strip()

email = """
Hi team,

Just a reminder that the quarterly planning meeting has moved from Thursday to Friday at 3 PM.
Please update your calendars and send me your project status slides by end of day Wednesday
so I can combine them into a single deck.

Thanks,
Priya
"""

print(suggest_subject(email))